In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/saas_50k_v1.csv")
df.head()

,account_id,month,company_size,industry,contract_type,discount_pct,regime_state,active_users,usage_growth,feature_adoption_rate,error_rate,tickets_count,ticket_growth,payment_delay_flag,current_mrr,next_month_mrr
0,0,1,SMB,Energy,Monthly,0.054167,stable,4.765884,0.017574,0.002766,0.102689,0.0,0.0,0,112.703497,50.000000
1,0,2,SMB,Energy,Monthly,0.054167,stable,11.706351,0.027244,0.000000,0.102174,2.0,inf,1,50.000000,51.903945
2,0,3,SMB,Energy,Monthly,0.054167,decline,12.706351,0.004655,0.082195,0.195298,2.0,0.0,0,51.903945,50.000000
3,0,4,SMB,Energy,Monthly,0.054167,decline,17.393579,-0.024496,0.062898,0.104958,3.0,0.5,1,50.000000,55.680067
4,0,5,SMB,Energy,Monthly,0.054167,decline,20.607523,-0.073376,0.120356,0.076803,0.0,-1.0,0,55.680067,50.000000


# Data Understanding

## 1. Time-series per account
- every (account_id) has a records for each (month)
- granularity = Account × Month

### a. Identifiers:
- account_id  
- month

### b. Semi-static:
- company_size
- industry
- contract_type
- regime_state

### c. Behavior:
- active_users
- usage_growth
- feature_adoption_rate

### d. Issues:
- error_rate
- tickets_count
- ticket_growth

### e. Revenue:
- discount_pct
- current_mrr
- next_month_mrr
- payment_delay_flag

## 2. Structure

In [2]:
df.shape
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1150000 entries, 0 to 1149999
Data columns (total 16 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   account_id             1150000 non-null  int64  
 1   month                  1150000 non-null  int64  
 2   company_size           1150000 non-null  str    
 3   industry               1150000 non-null  str    
 4   contract_type          1150000 non-null  str    
 5   discount_pct           1034760 non-null  float64
 6   regime_state           1150000 non-null  str    
 7   active_users           1150000 non-null  float64
 8   usage_growth           1150000 non-null  float64
 9   feature_adoption_rate  1115624 non-null  float64
 10  error_rate             1133821 non-null  float64
 11  tickets_count          1127186 non-null  float64
 12  ticket_growth          1150000 non-null  float64
 13  payment_delay_flag     1150000 non-null  int64  
 14  current_mrr            115000

In [3]:
# Descriptive statistics of the dataset
df.describe()

/home/youssef/ml_env/lib/python3.12/site-packages/pandas/core/nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,account_id,month,discount_pct,active_users,usage_growth,feature_adoption_rate,error_rate,tickets_count,ticket_growth,payment_delay_flag,current_mrr,next_month_mrr
count,1.150000e+06,1.150000e+06,1.034760e+06,1.150000e+06,1.150000e+06,1.115624e+06,1.133821e+06,1.127186e+06,1150000.00,1.150000e+06,1.150000e+06,1.150000e+06
mean,2.499950e+04,1.200000e+01,5.830647e-02,8.971999e+01,1.280262e-02,5.000573e-01,1.030176e-01,4.898935e+00,inf,2.718591e-01,3.218036e+02,2.897038e+02
std,1.443376e+04,6.633252e+00,3.611598e-02,7.613516e+01,3.688886e-02,3.397232e-01,6.704321e-02,4.205786e+00,NaN,4.449179e-01,1.530251e+03,1.597272e+03
min,0.000000e+00,1.000000e+00,2.853035e-04,1.000000e+00,-1.491025e-01,0.000000e+00,7.316996e-05,0.000000e+00,-1.00,0.000000e+00,2.001562e+01,5.000000e+01
25%,1.249975e+04,6.000000e+00,3.065270e-02,3.451877e+01,-1.202793e-02,1.650163e-01,5.217631e-02,2.000000e+00,-0.30,0.000000e+00,5.163699e+01,5.129363e+01
50%,2.499950e+04,1.200000e+01,5.178160e-02,7.052014e+01,1.437472e-02,4.999695e-01,8.996910e-02,4.000000e+00,0.00,0.000000e+00,6.913514e+01,6.783751e+01
75%,3.749925e+04,1.800000e+01,7.919370e-02,1.223813e+02,3.881885e-02,8.350957e-01,1.398397e-01,7.000000e+00,0.75,1.000000e+00,1.404441e+02,1.282552e+02
max,4.999900e+04,2.300000e+01,2.525988e-01,9.909708e+02,1.680977e-01,1.000000e+00,6.461280e-01,4.300000e+01,inf,1.000000e+00,1.716090e+05,2.691279e+05


## 3. Data Types Separation

In [4]:
# Duplicate check for account_id and month
df.duplicated(subset=['account_id', 'month']).sum()

np.int64(0)

## 4. Missing Values

In [5]:
# Check for missing values in the dataset
df.isnull().sum().sort_values(ascending=False)

discount_pct             115240
feature_adoption_rate     34376
tickets_count             22814
error_rate                16179
industry                      0
account_id                    0
company_size                  0
month                         0
active_users                  0
regime_state                  0
contract_type                 0
usage_growth                  0
ticket_growth                 0
payment_delay_flag            0
current_mrr                   0
next_month_mrr                0
dtype: int64

# Data Processing

## Template:
- Value Counts
- Data Type
- Missing Values
- Uniqueness
- Zero / Dominant Values
- Value Range
- Distribution
- Outliers
- Logical Consistency
- Decision

## a. Identifiers

### account_id

In [6]:
# 1.check integer type for account_id and convert to numeric if necessary
df["account_id"] = pd.to_numeric(df["account_id"], errors="coerce").astype("Int64")

# 2. remove rows with missing or invalid account_id values
df = df[df["account_id"].notna()]
df = df[df["account_id"] >= 0]

### month

### more checks

In [7]:
# Check for duplicates for account_id and month
duplicates = df[df.duplicated(subset=["account_id", "month"], keep=False)]

print("Duplicates:")
print("Number of duplicate rows:", len(duplicates))

if len(duplicates) > 0:
    print(duplicates[["account_id", "month"]].head())

Duplicates:
Number of duplicate rows: 0


In [8]:
print("\nPeriod Conversion Check:")

if str(df["month"].dtype).startswith("period"):
    print("Can convert to Period")
else:
    try:
        pd.to_datetime(df["month"]).dt.to_period("M")
        print("Can convert to Period")
    except Exception as e:
        print("Cannot convert to Period", e)


Period Conversion Check:
Can convert to Period


## b. Semi Static

### 1. Data Type

In [9]:
cols = ["company_size", "industry", "contract_type", "regime_state"]

print("Data Types:")
print(df[cols].dtypes)

Data Types:
company_size     str
industry         str
contract_type    str
regime_state     str
dtype: object


### 2. Missing Values

In [10]:
print("\nMissing Values:")
print(df[cols].isna().sum())


Missing Values:
company_size     0
industry         0
contract_type    0
regime_state     0
dtype: int64


### 3. Decision

In [11]:
for col in cols:
    nunique = df[col].nunique()
    missing = df[col].isna().sum()
    
    print(f"\n{col}:")
    print("unique values:", nunique)
    print("missing:", missing)


company_size:
unique values: 3
missing: 0

industry:
unique values: 10
missing: 0

contract_type:
unique values: 2
missing: 0

regime_state:
unique values: 3
missing: 0


## c. Behavior

### 1. Value Counts

In [12]:
behavior_cols = ["active_users", "usage_growth", "feature_adoption_rate"]

for col in behavior_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).head(10))


active_users
active_users
1.0     2398
2.0      701
3.0      342
4.0      198
5.0      127
6.0       87
7.0       58
8.0       47
9.0       35
10.0      28
Name: count, dtype: int64

usage_growth
usage_growth
 0.017574    1
 0.027244    1
 0.004655    1
-0.024496    1
-0.073376    1
-0.039397    1
-0.029551    1
-0.009002    1
 0.004810    1
-0.006846    1
Name: count, dtype: int64

feature_adoption_rate
feature_adoption_rate
1.000000    46387
0.000000    46211
NaN         34376
0.002766        1
0.082195        1
0.062898        1
0.120356        1
0.182621        1
0.200939        1
0.292003        1
Name: count, dtype: int64


### 2. Missing Values

In [13]:
for col in behavior_cols:
    print(f"\n{col}")
    print("Missing Count:", df[col].isna().sum())
    print("Missing %:", df[col].isna().mean())


active_users
Missing Count: 0
Missing %: 0.0

usage_growth
Missing Count: 0
Missing %: 0.0

feature_adoption_rate
Missing Count: 34376
Missing %: 0.02989217391304348


### 3. Uniqueness

In [14]:
for col in behavior_cols:
    print(f"{col} Unique Values:", df[col].nunique())

active_users Unique Values: 1145882
usage_growth Unique Values: 1150000
feature_adoption_rate Unique Values: 1023028


### 4. Zero / Dominant Values

In [15]:
for col in behavior_cols:
    print(f"\n{col}")
    print("Zero %:", (df[col] == 0).mean())
    print("Top Values:")
    print(df[col].value_counts(normalize=True).head(3))


active_users
Zero %: 0.0
Top Values:
active_users
1.0    0.002085
2.0    0.000610
3.0    0.000297
Name: proportion, dtype: float64

usage_growth
Zero %: 0.0
Top Values:
usage_growth
0.017574    8.695652e-07
0.027244    8.695652e-07
0.004655    8.695652e-07
Name: proportion, dtype: float64

feature_adoption_rate
Zero %: 0.040183478260869564
Top Values:
feature_adoption_rate
1.000000    4.157942e-02
0.000000    4.142166e-02
0.002766    8.963593e-07
Name: proportion, dtype: float64


### 5. Value Range

In [16]:
for col in behavior_cols:
    print(f"{col} Range:", df[col].min(), df[col].max())

active_users Range: 1.0 990.9708491797354
usage_growth Range: -0.1491024658529373 0.1680976721895077
feature_adoption_rate Range: 0.0 1.0


### 6. Distribution

In [17]:
for col in behavior_cols:
    print(f"\n{col}")
    print(df[col].describe())


active_users
count    1.150000e+06
mean     8.971999e+01
std      7.613516e+01
min      1.000000e+00
25%      3.451877e+01
50%      7.052014e+01
75%      1.223813e+02
max      9.909708e+02
Name: active_users, dtype: float64

usage_growth
count    1.150000e+06
mean     1.280262e-02
std      3.688886e-02
min     -1.491025e-01
25%     -1.202793e-02
50%      1.437472e-02
75%      3.881885e-02
max      1.680977e-01
Name: usage_growth, dtype: float64

feature_adoption_rate
count    1.115624e+06
mean     5.000573e-01
std      3.397232e-01
min      0.000000e+00
25%      1.650163e-01
50%      4.999695e-01
75%      8.350957e-01
max      1.000000e+00
Name: feature_adoption_rate, dtype: float64


### 7. Logical Consistency

In [18]:
# active_users
print("active_users < 0:", (df["active_users"] < 0).sum())

# usage_growth
print("usage_growth < -1:", (df["usage_growth"] < -1).sum())
print("usage_growth > 5:", (df["usage_growth"] > 5).sum())

# feature_adoption_rate
print("feature_adoption_rate < 0:", (df["feature_adoption_rate"] < 0).sum())
print("feature_adoption_rate > 1:", (df["feature_adoption_rate"] > 1).sum())

active_users < 0: 0
usage_growth < -1: 0
usage_growth > 5: 0
feature_adoption_rate < 0: 0
feature_adoption_rate > 1: 0


### 8. Decision

In [19]:
df["active_users"] = df["active_users"].astype(int)
df["feature_adoption_rate"] = df["feature_adoption_rate"].fillna(0)

## d. Issues

In [20]:
issues_cols = ["error_rate", "tickets_count", "ticket_growth"]

### 1. Value Counts

In [21]:
for col in issues_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(10))


error_rate:
error_rate
NaN         16179
0.102689        1
0.102174        1
0.195298        1
0.104958        1
0.076803        1
0.255628        1
0.121391        1
0.006810        1
0.221271        1
Name: count, dtype: int64

tickets_count:
tickets_count
2.0    139764
3.0    138680
4.0    126770
1.0    124879
5.0    107536
0.0    101854
6.0     87286
7.0     69334
8.0     53579
9.0     40289
Name: count, dtype: int64

ticket_growth:
ticket_growth
 0.000000    229830
 1.000000     70643
 inf          68977
-0.500000     57990
 0.500000     46167
-1.000000     43442
-0.333333     40411
 2.000000     34391
 0.333333     30664
-0.250000     27705
Name: count, dtype: int64


### 2. Data Type

In [22]:
for col in issues_cols:
    print(f"{col} → {df[col].dtype}")

error_rate → float64
tickets_count → float64
ticket_growth → float64


### 3. Missing Values

In [23]:
for col in issues_cols:
    print(f"\n{col}")
    print("Missing Count:", df[col].isna().sum())
    print("Missing %:", df[col].isna().mean())


error_rate
Missing Count: 16179
Missing %: 0.014068695652173913

tickets_count
Missing Count: 22814
Missing %: 0.01983826086956522

ticket_growth
Missing Count: 0
Missing %: 0.0


### 4. Uniqueness

In [24]:
for col in issues_cols:
    print(f"{col} Unique Values:", df[col].nunique())

error_rate Unique Values: 1133821
tickets_count Unique Values: 42
ticket_growth Unique Values: 576


### 5. Zero / Dominant Values

In [25]:
for col in issues_cols:
    print(f"\n{col}")
    print("Zero %:", (df[col] == 0).mean())
    print("Top Values:")
    print(df[col].value_counts(normalize=True).head(3))


error_rate
Zero %: 0.0
Top Values:
error_rate
0.102689    8.819734e-07
0.102174    8.819734e-07
0.195298    8.819734e-07
Name: proportion, dtype: float64

tickets_count
Zero %: 0.08856869565217391
Top Values:
tickets_count
2.0    0.123994
3.0    0.123032
4.0    0.112466
Name: proportion, dtype: float64

ticket_growth
Zero %: 0.19985217391304347
Top Values:
ticket_growth
0.0    0.199852
1.0    0.061429
inf    0.059980
Name: proportion, dtype: float64


### 6. Logical Consistency

In [26]:
# error_rate
print("error_rate < 0:", (df["error_rate"] < 0).sum())
print("error_rate > 1:", (df["error_rate"] > 1).sum())

# tickets_count (count: >= 0)
print("tickets_count < 0:", (df["tickets_count"] < 0).sum())

# ticket_growth
print("ticket_growth < -1:", (df["ticket_growth"] < -1).sum())
print("ticket_growth > 5:", (df["ticket_growth"] > 5).sum())

error_rate < 0: 0
error_rate > 1: 0
tickets_count < 0: 0
ticket_growth < -1: 0
ticket_growth > 5: 75501


### 7. Decision

In [27]:
# Remove missing values and clip error_rate to be between 0 and 1
df["error_rate"] = df["error_rate"].fillna(0)
df["error_rate"] = df["error_rate"].clip(0, 1)

In [28]:
# Remove missing values and convert tickets_count to integer
df["tickets_count"] = df["tickets_count"].fillna(0)
df["tickets_count"] = df["tickets_count"].astype(int)

In [29]:
# the problem is "growth = (new - old) / old" may equal inf when old=0
df["ticket_growth"] = df["ticket_growth"].replace([np.inf, -np.inf], np.nan)
df["ticket_growth"] = df["ticket_growth"].fillna(0)
df["ticket_growth"] = df["ticket_growth"].clip(-1, 3)

## e. Revenue

In [30]:
revenue_cols = [
    "discount_pct",
    "current_mrr",
    "next_month_mrr",
    "payment_delay_flag"
]

### 1. Value Counts

In [31]:
for col in revenue_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(10))


discount_pct:
discount_pct
NaN         115240
0.114037        23
0.107310        23
0.053770        23
0.090515        23
0.059635        23
0.014771        23
0.043387        23
0.037433        23
0.057000        23
Name: count, dtype: int64

current_mrr:


current_mrr
50.000000     261268
112.703497         1
51.903945          1
55.680067          1
59.779118          1
53.771521          1
65.528626          1
54.989430          1
51.548690          1
52.928285          1
Name: count, dtype: int64

next_month_mrr:
next_month_mrr
50.000000    268583
51.903945         1
55.680067         1
59.779118         1
53.771521         1
65.528626         1
54.989430         1
51.548690         1
52.928285         1
51.945708         1
Name: count, dtype: int64

payment_delay_flag:
payment_delay_flag
0    837362
1    312638
Name: count, dtype: int64


### 2. Data Type

In [32]:
for col in revenue_cols:
    print(f"{col}: {df[col].dtype}")

discount_pct: float64
current_mrr: float64
next_month_mrr: float64
payment_delay_flag: int64


### 3. Zero / Dominant Values

In [33]:
for col in revenue_cols:
    print(f"\n{col}:")
    
    if df[col].dtype != 'object':
        print("Zeros:", (df[col] == 0).sum())
    
    print("Top Values:")
    print(df[col].value_counts(normalize=True).head())


discount_pct:
Zeros: 0
Top Values:
discount_pct
0.114037    0.000022
0.107310    0.000022
0.053770    0.000022
0.090515    0.000022
0.059635    0.000022
Name: proportion, dtype: float64

current_mrr:
Zeros: 0
Top Values:
current_mrr
50.000000     2.271896e-01
112.703497    8.695652e-07
51.903945     8.695652e-07
55.680067     8.695652e-07
59.779118     8.695652e-07
Name: proportion, dtype: float64

next_month_mrr:
Zeros: 0
Top Values:
next_month_mrr
50.000000    2.335504e-01
51.903945    8.695652e-07
55.680067    8.695652e-07
59.779118    8.695652e-07
53.771521    8.695652e-07
Name: proportion, dtype: float64

payment_delay_flag:
Zeros: 837362
Top Values:
payment_delay_flag
0    0.728141
1    0.271859
Name: proportion, dtype: float64


### 4. Value Range

In [34]:
for col in revenue_cols:
    if df[col].dtype != 'object':
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")

discount_pct: min=0.0002853034818894, max=0.2525988146704591
current_mrr: min=20.01561814836349, max=171608.97115296312
next_month_mrr: min=50.0, max=269127.8956913708
payment_delay_flag: min=0, max=1


### 5. Outliers

In [35]:
for col in revenue_cols:
    if df[col].dtype != 'object':
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        
        outliers = df[(df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)]
        
        print(f"{col}: outliers = {len(outliers)}")

discount_pct: outliers = 18302
current_mrr: outliers = 168849
next_month_mrr: outliers = 165650
payment_delay_flag: outliers = 0


### 6. Logical Consistency

In [36]:
# Check for invalid discount percentages
df[(df["discount_pct"] < 0) | (df["discount_pct"] > 100)]

# Current and Next Month MRR
df[df["current_mrr"] < 0]
df[df["next_month_mrr"] < 0]
df[df["next_month_mrr"] < df["current_mrr"]]

,account_id,month,company_size,industry,contract_type,discount_pct,regime_state,active_users,usage_growth,feature_adoption_rate,error_rate,tickets_count,ticket_growth,payment_delay_flag,current_mrr,next_month_mrr
0,0,1,SMB,Energy,Monthly,0.054167,stable,4,0.017574,0.002766,0.102689,0,0.00,0,112.703497,50.000000
2,0,3,SMB,Energy,Monthly,0.054167,decline,12,0.004655,0.082195,0.195298,2,0.00,0,51.903945,50.000000
4,0,5,SMB,Energy,Monthly,0.054167,decline,20,-0.073376,0.120356,0.076803,0,-1.00,0,55.680067,50.000000
6,0,7,SMB,Energy,Monthly,0.054167,stable,26,-0.029551,0.200939,0.121391,1,-0.75,0,59.779118,50.000000
9,0,10,SMB,Energy,Monthly,0.054167,decline,37,-0.006846,0.302729,0.481068,2,-0.50,0,53.771521,50.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1149981,49999,5,Mid,Manufacturing,Monthly,0.107606,stable,35,-0.001196,0.182758,0.075192,1,0.00,0,50.980568,50.000000
1149983,49999,7,Mid,Manufacturing,Monthly,0.107606,growth,57,0.096434,0.173217,0.140057,2,-0.50,0,58.622493,50.000000
1149985,49999,9,Mid,Manufacturing,Monthly,0.107606,stable,79,0.036825,0.242654,0.069190,4,0.00,0,54.408061,50.000000
1149992,49999,16,Mid,Manufacturing,Monthly,0.107606,stable,143,-0.007413,0.837037,0.138355,3,-0.25,0,119.367002,54.007702


### 7. Decision

In [37]:
df["discount_pct"] = df["discount_pct"].fillna(0)

# Final Check

In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1150000 entries, 0 to 1149999
Data columns (total 16 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   account_id             1150000 non-null  Int64  
 1   month                  1150000 non-null  int64  
 2   company_size           1150000 non-null  str    
 3   industry               1150000 non-null  str    
 4   contract_type          1150000 non-null  str    
 5   discount_pct           1150000 non-null  float64
 6   regime_state           1150000 non-null  str    
 7   active_users           1150000 non-null  int64  
 8   usage_growth           1150000 non-null  float64
 9   feature_adoption_rate  1150000 non-null  float64
 10  error_rate             1150000 non-null  float64
 11  tickets_count          1150000 non-null  int64  
 12  ticket_growth          1150000 non-null  float64
 13  payment_delay_flag     1150000 non-null  int64  
 14  current_mrr            115000

In [39]:
df.isnull().sum().sort_values(ascending=False)

account_id               0
month                    0
company_size             0
industry                 0
contract_type            0
discount_pct             0
regime_state             0
active_users             0
usage_growth             0
feature_adoption_rate    0
error_rate               0
tickets_count            0
ticket_growth            0
payment_delay_flag       0
current_mrr              0
next_month_mrr           0
dtype: int64

In [ ]:
# Save the cleaned dataset to a new file
df.to_parquet(
    "data/cleaned_data.parquet",
    compression="gzip")

In [ ]:
# Split the DataFrame into chunks
chunk_size = len(df) // 2

df.iloc[:chunk_size].to_parquet("data/data_part_0.parquet", compression="gzip")
df.iloc[chunk_size:].to_parquet("data/data_part_1.parquet", compression="gzip")